In [32]:
import pandas as pd
import numpy as np

df = pd.read_parquet(
    "../data/processed/papers_clean.parquet"
)

print(df.shape)

(876, 18)


In [33]:
df["text_for_embedding"] = (
    df["title"].fillna("")
    + ". "
    + df["abstract"].fillna("")
)

df["text_for_embedding"].head()

0    A Comprehensive Overview of Large Language Mod...
1    A Comprehensive Survey of Hallucination Mitiga...
2    A Comprehensive Survey of Retrieval-Augmented ...
3    A Comprehensive Survey of Retrieval-Augmented ...
4    A Comprehensive Survey on Multimodal RAG: All ...
Name: text_for_embedding, dtype: str

In [34]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6465.43it/s]


In [35]:
valid_mask = df["abstract"].fillna("").str.strip().ne("")
embedded_df = df.loc[valid_mask].copy().reset_index(drop=True)

texts = df.loc[
    valid_mask,
    "text_for_embedding"
].tolist()

print("Texts to embed:", len(texts))

Texts to embed: 767


In [36]:
embeddings = model.encode(
    texts,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

Batches: 100%|██████████| 24/24 [00:03<00:00,  6.53it/s]

Embedding shape: (767, 384)


In [37]:
import numpy as np

np.save(
    "../data/processed/paper_embeddings.npy",
    embeddings
)

paper_ids = df.loc[
    valid_mask,
    "paper_id"
].to_numpy(dtype=str)

np.save(
    "../data/processed/embedded_paper_ids.npy",
    paper_ids
)

print("Embeddings saved successfully.")

Embeddings saved successfully.


In [38]:
from sklearn.metrics.pairwise import cosine_similarity

In [47]:
def search_papers(query, top_k=10):
    """
    Search the embedded papers using semantic similarity.

    Parameters:
        query (str): Natural-language search query
        top_k (int): Number of results to return

    Returns:
        DataFrame containing the most similar papers
    """

    # Convert query into an embedding
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )

    # Calculate cosine similarity against all papers
    similarities = cosine_similarity(
        query_embedding,
        embeddings
    )[0]

    # Get indices of the most similar papers
    top_indices = np.argsort(
        similarities
    )[::-1][:top_k]

    # Build results
    results = embedded_df.iloc[
        top_indices
    ][
        [
            "title",
            "paper_id",
            "doi",
            "year",
            "citation_count"
        ]
    ].copy()

    # Add similarity score
    results["similarity"] = similarities[top_indices]

    # Reset index for clean output
    results = results.reset_index(drop=True)

    return results

In [48]:
search_papers(
    "graph based retrieval augmented generation",
    top_k=10
)

,title,paper_id,doi,year,citation_count,similarity
0,Knowledge Graph-Guided Retrieval Augmented Gen...,https://openalex.org/W4411119292,https://doi.org/10.18653/v1/2025.naacl-long.449,2025,76,0.800015
1,SeaKR: Self-aware Knowledge Retrieval for Adap...,https://openalex.org/W4412889884,https://doi.org/10.18653/v1/2025.acl-long.1312,2025,24,0.759055
2,Searching for Best Practices in Retrieval-Augm...,https://openalex.org/W4404782883,https://doi.org/10.18653/v1/2024.emnlp-main.981,2024,109,0.734508
3,Graph Retrieval-Augmented Generation: A Survey,https://openalex.org/W4416379250,https://doi.org/10.1145/3777378,2025,145,0.708417
4,"Fact, Fetch, and Reason: A Unified Evaluation ...",https://openalex.org/W4411119426,https://doi.org/10.18653/v1/2025.naacl-long.243,2025,21,0.701093
5,A Survey on Retrieval-Augmented Text Generation,https://openalex.org/W4226069413,https://doi.org/10.48550/arxiv.2202.01110,2022,71,0.699357
6,GFM-RAG: Graph Foundation Model for Retrieval ...,https://openalex.org/W4407129024,https://doi.org/10.52202/085713-1222,2025,7,0.697280
7,Active Retrieval Augmented Generation,https://openalex.org/W4389519118,https://doi.org/10.18653/v1/2023.emnlp-main.495,2023,454,0.694989
8,Knowledge Graph Retrieval-Augmented Generation...,https://openalex.org/W4412889887,https://doi.org/10.18653/v1/2025.acl-long.1317,2025,55,0.694590
9,D-RAG: Differentiable Retrieval-Augmented Gene...,https://openalex.org/W4416035020,https://doi.org/10.18653/v1/2025.emnlp-main.1793,2025,5,0.690017


In [49]:
import numpy as np

np.save(
    "../data/processed/paper_embeddings.npy",
    embeddings
)

print("Embeddings saved:", embeddings.shape)

Embeddings saved: (767, 384)


In [51]:
embedded_df.to_parquet(
    "../data/processed/embedded_papers.parquet",
    index=False
)

print("Embedded paper metadata saved:", embedded_df.shape)

Embedded paper metadata saved: (767, 19)
